In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path('..').resolve()
FISHER_TAB = ROOT / 'results' / 'fisher_day0_day15' / 'tables'
POGO_TAB = ROOT / 'pogorelyy_repro' / 'results' / 'tables'
OUT = ROOT / 'coursework_submission_notebooks' / 'outputs' / '04_compare'
TAB = OUT / 'tables'
FIG = OUT / 'figures'
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)
DONORS = ['P1', 'P2', 'Q1', 'Q2', 'S1', 'S2']


In [ ]:
summary_rows = []
union_rows = []
for donor in DONORS:
    f = pd.read_csv(FISHER_TAB / f'fisher_positive_FC4_{donor}.csv')
    m = pd.read_csv(POGO_TAB / f'{donor}_15_top_expanded.tsv', sep='\t')
    Fs = set(f['cdr3_nt'].astype(str))
    Ms = set(m['cdr3_nt'].astype(str))
    both = Fs & Ms
    fo = Fs - Ms
    mo = Ms - Fs
    union = Fs | Ms
    jaccard = len(both) / len(union) if union else np.nan
    summary_rows.append({'donor': donor, 'n_fisher_FC4': len(Fs), 'n_model': len(Ms), 'n_fisher_only': len(fo), 'n_both': len(both), 'n_model_only': len(mo), 'n_union': len(union), 'jaccard': jaccard, 'pct_fisher_also_model': 100.0 * len(both) / len(Fs) if Fs else np.nan, 'pct_model_also_fisher': 100.0 * len(both) / len(Ms) if Ms else np.nan})
    f_idx = f.set_index('cdr3_nt')
    m_idx = m.set_index('cdr3_nt')
    for (nt, group) in [(x, 'Fisher only') for x in fo] + [(x, 'Both') for x in both] + [(x, 'Model only') for x in mo]:
        row = {'donor': donor, 'cdr3_nt': nt, 'overlap_group': group}
        if nt in f_idx.index:
            fr = f_idx.loc[nt]
            if isinstance(fr, pd.DataFrame):
                fr = fr.iloc[0]
            row['cdr3_aa'] = str(fr.cdr3_aa)
            row['fold_change_fisher'] = float(fr.fold_change)
            row['q_value'] = float(fr.q_value)
        if nt in m_idx.index:
            mr = m_idx.loc[nt]
            if isinstance(mr, pd.DataFrame):
                mr = mr.iloc[0]
            row['cdr3_aa'] = str(mr.cdr3_aa) if 'cdr3_aa' not in row else row['cdr3_aa']
            row['s_med'] = float(mr.s_med)
            row['p_nonexpansion'] = float(mr.p_nonexpansion)
        union_rows.append(row)
summary = pd.DataFrame(summary_rows)
union = pd.DataFrame(union_rows)
summary.to_csv(TAB / 'overlap_summary.csv', index=False)
union.to_csv(TAB / 'union_clonotypes.csv', index=False)
summary


In [ ]:
(fig, axes) = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(summary))
w = 0.35
axes[0].bar(x - w / 2, summary['n_fisher_FC4'], width=w, color='#C0392B', label='Fisher')
axes[0].bar(x + w / 2, summary['n_model'], width=w, color='#2980B9', label='Model')
axes[0].set_xticks(x)
axes[0].set_xticklabels(summary['donor'])

axes[0].legend()
axes[1].bar(x, summary['n_fisher_only'], color='#E67E22', label='Fisher only')
axes[1].bar(x, summary['n_both'], bottom=summary['n_fisher_only'], color='#8E44AD', label='Both')
bottom2 = summary['n_fisher_only'] + summary['n_both']
axes[1].bar(x, summary['n_model_only'], bottom=bottom2, color='#1ABC9C', label='Model only')
axes[1].set_xticks(x)
axes[1].set_xticklabels(summary['donor'])

axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG / 'fisher_vs_model.png', dpi=150, bbox_inches='tight')
plt.show()
print(summary[['donor', 'jaccard']].to_string(index=False))
